# Film 5 — minimalna długość hasła jako decyzja, nie stała

Notebook **rozmawia z systemem stojącym w Dockerze** (nic nie stubuje), przechodzi całą sekwencję
filmu scena po scenie, **sprawdza każdy krok asercją** i **robi zrzuty ekranu** galerii.

Wymaga działającego stacku:

```bash
cd ~/Documents/git/portal && ./infra-up.sh
```

Uruchom `Run All`. Na końcu jest tabela PASS/FAIL i lista zrzutów w `shots/`.
Notebook sprząta po sobie: kasuje wiersz `security_settings` i zdejmuje override compose'a.

In [ ]:
# --- configuration and helpers -------------------------------------------------
import json, re, subprocess, time, pathlib, datetime
import requests
from IPython.display import HTML, Image, display

SEC      = "http://localhost:8080"
MEMES    = "http://localhost:8083"
MAILPIT  = "http://localhost:8025"
PROJECT  = "security"                      # one compose project for the whole estate
PORTAL   = pathlib.Path.home() / "Documents/git/portal"
HERE     = pathlib.Path.cwd()
SHOTS    = HERE / "shots"; SHOTS.mkdir(exist_ok=True)

ADMIN    = "admin@example.com"
PASSWORD = "StrongPassword1!"              # spełnia każdą regułę polityki
KEY      = "security.password.policy.min.length"
RUN      = datetime.datetime.now().strftime("%H%M%S")   # fresh e-mail addresses per run

CHECKS = []
def check(name, ok, detail=""):
    CHECKS.append((name, bool(ok), str(detail)[:300])); 
    display(HTML(f"<div style='font-family:monospace;padding:2px 0'>"
                 f"<b style='color:{'#137333' if ok else '#c5221f'}'>{'PASS' if ok else 'FAIL'}</b> "
                 f"{name} <span style='color:#666'>{detail if not ok else ''}</span></div>"))
    return ok

def sh(*args, cwd=None, timeout=180):
    """Run a command and return its output; a non-zero exit is data, not an exception."""
    p = subprocess.run(args, cwd=cwd, capture_output=True, text=True, timeout=timeout)
    return (p.stdout + p.stderr).strip()

def psql(sql):
    """SQL straight against the security database - the way someone at the console writes it."""
    return sh("docker", "compose", "-p", PROJECT, "exec", "-T", "postgres",
              "psql", "-U", "postgres", "-d", "security", "-c", sql)

def show(title, response):
    """Render an HTTP response legibly enough to be read on camera."""
    try:    body = json.dumps(response.json(), indent=2, ensure_ascii=False)
    except Exception: body = response.text or "(puste ciało)"
    color = "#137333" if response.status_code < 400 else "#c5221f"
    display(HTML(f"<div style='font-family:monospace;font-size:13px;border-left:3px solid {color};"
                 f"padding:4px 10px;margin:4px 0;background:#fafafa'>"
                 f"<b>{title}</b> &rarr; <b style='color:{color}'>{response.status_code}</b>"
                 f"<pre style='margin:4px 0'>{body}</pre></div>"))
    return response

print("gotowe — SEC:", SEC, "| katalog zrzutów:", SHOTS)

## 0. Preflight — czy system w ogóle stoi

In [ ]:
def health():
    rows = []
    for name, url in [("security (readiness)", f"{SEC}/health/readiness"),
                      ("memes / galeria",      f"{MEMES}/"),
                      ("Mailpit",              f"{MAILPIT}/")]:
        try:    code_ = requests.get(url, timeout=5).status_code
        except Exception as e: code_ = f"brak ({type(e).__name__})"
        rows.append((name, url, code_))
    db = psql("select 1;")
    rows.append(("postgres (security)", "docker compose exec postgres", "OK" if "1 row" in db else db[:80]))
    html = "".join(f"<tr><td style='padding:2px 12px 2px 0'>{n}</td>"
                   f"<td style='padding:2px 12px 2px 0;color:#666'>{u}</td>"
                   f"<td><b>{c}</b></td></tr>" for n, u, c in rows)
    display(HTML(f"<table style='font-family:monospace;font-size:13px'>{html}</table>"))
    return rows

rows = health()
check("stack odpowiada", all(str(c) in ("200", "OK") for _, _, c in rows),
      "podnieś stack: cd ~/Documents/git/portal && ./infra-up.sh")

## 1. Czysty start — kasujemy wiersz z poprzednich prób

Drabinka ma trzy szczeble: **live (baza) > restart (property) > rebuild (`MinLength.DEFAULT` = 5)**.
Pusty szczebel live oznacza, że film zaczyna się od wartości domyślnej.

In [ ]:
print(psql(f"DELETE FROM security_settings WHERE name = '{KEY}';"))
print(psql("SELECT * FROM security_settings;"))
check("szczebel live jest pusty", "(0 rows)" in psql("SELECT * FROM security_settings;"))

## 2. Konto ADMIN-a — rejestracja, link z Mailpita, weryfikacja, logowanie

In [ ]:
def mail_token(to, tries=20):
    """Pull the token out of the latest mail to this address (Mailpit is the dev inbox)."""
    for _ in range(tries):
        msgs = requests.get(f"{MAILPIT}/api/v1/search", params={"query": f"to:{to}"}, timeout=5).json()["messages"]
        if msgs:
            text = requests.get(f"{MAILPIT}/api/v1/message/{msgs[0]['ID']}", timeout=5).json()["Text"]
            m = re.search(r"(?:token|verify)=([A-Za-z0-9_\-]+)", text)
            if m: return m.group(1)
        time.sleep(1)
    return None

def register(email, password): return requests.post(f"{SEC}/register", json={"email": email, "password": password}, timeout=15)
def authenticate(email, password): return requests.post(f"{SEC}/authenticate", json={"email": email, "password": password}, timeout=15)

def ensure_admin():
    """The ADMIN is designated in compose (SECURITY_BOOTSTRAP_ADMINS); here we only create the account."""
    r = authenticate(ADMIN, PASSWORD)
    if r.status_code == 200:
        return r.json()["accessToken"]
    register(ADMIN, PASSWORD)                       # 201 for a taken address too (anti-enumeration)
    token = mail_token(ADMIN)
    if token:
        requests.post(f"{SEC}/verify-email", json={"token": token}, timeout=15)
    r = authenticate(ADMIN, PASSWORD)
    return r.json()["accessToken"] if r.status_code == 200 else None

TOKEN = ensure_admin()
AUTH  = {"Authorization": f"Bearer {TOKEN}"}
check("ADMIN zalogowany", TOKEN is not None, "sprawdź SECURITY_BOOTSTRAP_ADMINS w compose")

## 3. Raport drabinki i ręka ADMIN-a

`GET /admin/settings/password/min-length` mówi nie tylko **ile**, ale **kto odpowiedział** i **co po drodze
odrzucono**. `POST` idzie przez step-up — polityka wiąże każde przyszłe hasło, więc żywa sesja to za mało.

In [ ]:
def report():
    return show("GET /admin/settings/password/min-length",
                requests.get(f"{SEC}/admin/settings/password/min-length", headers=AUTH, timeout=15)).json()

def step_up():
    return requests.post(f"{SEC}/account/step-up", headers=AUTH,
                         json={"action": "admin-settings", "password": PASSWORD}, timeout=15)

def set_min_length(value):
    """An elevation is one-shot and per action - buy a fresh one right before each write."""
    assert step_up().status_code == 200, "step-up nie przeszedł"
    return show(f"POST min-length = {value}",
                requests.post(f"{SEC}/admin/settings/password/min-length", headers=AUTH,
                              json={"value": value}, timeout=15))

r = report()
check("w mocy jest 5 z domyślnego szczebla", r["value"] == 5 and "rebuild" in r["source"], r)

### Scena 1 — programista ustala domyślną wartość

Pięcioznakowe hasło (z cyfrą, wielką literą i znakiem specjalnym) przechodzi, bo w mocy jest 5.

In [ ]:
r = show("POST /register (hasło 5-znakowe)", register(f"u1-{RUN}@example.com", "Ab1!x"))
check("rejestracja 5-znakowym hasłem przechodzi", r.status_code == 201, r.text)

### Scena 2 — wdrożenie zajmuje szczebel „restart"

Property `security.password.policy.min.length` (tu jako zmienna środowiskowa) przykrywa domyślną
wartość — ale dopiero po restarcie serwisu. Notebook podnosi override compose'a i czeka na readiness.

In [ ]:
OVERRIDE = HERE / "compose.override.yml"

def restart_security(min_length=None):
    """None = no property (the starting state); a number = the deployment claims the restart rung."""
    if min_length is None:
        OVERRIDE.write_text("services:\n  security:\n    environment: {}\n")
    else:
        OVERRIDE.write_text("services:\n  security:\n    environment:\n"
                            f"      SECURITY_PASSWORD_POLICY_MIN_LENGTH: \"{min_length}\"\n")
    out = sh("docker", "compose", "-p", PROJECT, "-f", "docker-compose.yml", "-f", str(OVERRIDE),
             "up", "-d", "--no-build", "security", cwd=PORTAL, timeout=300)
    for _ in range(60):
        try:
            if requests.get(f"{SEC}/health/readiness", timeout=3).status_code == 200: return out
        except Exception: pass
        time.sleep(3)
    return out + " (readiness nie wrócił)"

restart_security(8)
r = report()
check("odpowiada szczebel restart, wartość 8", r["value"] == 8 and "restart" in r["source"], r)

resp = show("POST /register (hasło 7-znakowe)", register(f"u2-{RUN}@example.com", "Ab1!xyz"))
check("7 znaków odrzucone, komunikat niesie 8", resp.status_code == 422
      and any(e.get("MIN_LENGTH_NOT_MET") == 8 for e in resp.json()["passwordErrors"]), resp.text)

### Scena 3 — ADMIN decyduje w trakcie pracy systemu

Szczebel live przykrywa property. Od tej chwili każde miejsce, w którym ustala się hasło, mierzy nową miarą —
a odmowa **nazywa obowiązujące minimum**, bo polityka jest konfiguracją żywą i goły kod błędu zostawiłby
pytającego z domysłami.

In [ ]:
set_min_length(10)
r = report()
check("w mocy jest 10 ze szczebla live", r["value"] == 10 and "live" in r["source"], r)

resp = show("POST /register (hasło 9-znakowe)", register(f"u3-{RUN}@example.com", "Nine1!aaa"))
check("9 znaków odrzucone, komunikat niesie 10", resp.status_code == 422
      and any(e.get("MIN_LENGTH_NOT_MET") == 10 for e in resp.json()["passwordErrors"]), resp.text)

### Scena 4a — wartość poniżej własnej podłogi polityki

Value object `MinLength` jest jedyną bramką. Odmowa przechodzi przez use-case, więc **nic się nie zmienia**.

In [ ]:
resp = set_min_length(3)
check("3 odrzucone z powodem", resp.status_code == 400 and resp.json()["status"] == "REFUSED", resp.text)

r = report()
check("po odmowie w mocy nadal 10", r["value"] == 10 and "live" in r["source"], r)

### Scena 4b — wiersz wpisany prosto do bazy nie jest prawem

Zapis z konsoli psql omija bramkę — to naruszenie umowy, nie luka w projekcie (baza aplikacyjna wg Fowlera).
Drabinka odrzuca nielegalny szczebel, **spada stopień niżej** i mówi o tym w raporcie oraz w logu.

In [ ]:
print(psql(f"INSERT INTO security_settings (name, value) VALUES ('{KEY}', '3') "
           f"ON CONFLICT (name) DO UPDATE SET value = EXCLUDED.value, updated_at = now();"))
print(psql("SELECT name, value FROM security_settings;"))

r = report()
rejected = r.get("rejected", [])
check("drabinka spadła na property (8), bo live jest nielegalny",
      r["value"] == 8 and "restart" in r["source"], r)
check("raport nazywa odrzucony szczebel i powód",
      any(x["source"].startswith("live") and str(x["value"]) == "3" and "at least" in x["reason"] for x in rejected), rejected)

log = sh("docker", "compose", "-p", PROJECT, "logs", "--since", "3m", "security")
warn = [l for l in log.splitlines() if "min.length" in l.lower() and "warn" in l.lower()]
display(HTML("<pre style='font-size:12px;background:#fafafa;padding:6px'>" + ("\n".join(warn[-3:]) or "(brak linii WARN w oknie 3 min)") + "</pre>"))
check("serwis ostrzega w logu o odrzuconym wierszu", bool(warn), "brak WARN")

### Scena 4c — a bez property drabinka spada aż do wartości domyślnej

Zdejmujemy szczebel restart: nielegalny wiersz zostaje w bazie, a w mocy jest `MinLength.DEFAULT` = 5.

In [ ]:
restart_security(None)
r = report()
check("w mocy jest 5 z domyślnego szczebla, live nadal odrzucony",
      r["value"] == 5 and "rebuild" in r["source"] and any(x["source"].startswith("live") for x in r.get("rejected", [])), r)

resp = show("POST /register (hasło 5-znakowe)", register(f"u4-{RUN}@example.com", "Ab1!x"))
check("5 znaków znów przechodzi", resp.status_code == 201, resp.text)

### Scena 5 — to samo widziane z galerii

Przeglądarka pokazuje odmowę tak, jak zobaczy ją użytkownik: **pogrupowaną po polach**, z parametrem
polityki obowiązującym w tej właśnie próbie. Zrzuty lądują w `shots/`.

In [ ]:
# Jupyter already runs an asyncio loop, so Playwright uses the async API (top-level await).
from playwright.async_api import async_playwright

async def shot(page, name, width=760):
    path = SHOTS / f"{name}.png"
    await page.screenshot(path=str(path))
    display(Image(str(path), width=width))
    return path

pw      = await async_playwright().start()
browser = await pw.chromium.launch()
page    = await browser.new_page(viewport={"width": 1100, "height": 800}, device_scale_factor=2)

await page.goto(MEMES, wait_until="networkidle")
await shot(page, "01-galeria")

await page.get_by_role("tab", name="Create account").click()
# an address the browser lets through (type="email") but the server refuses - so BOTH sections show
await page.get_by_label("e-mail").fill(f"ui-{RUN}@wp")
await page.get_by_label("password").fill("abc")                    # breaks several rules at once
await page.get_by_role("button", name="Create account").click()
await page.wait_for_selector("text=That will not do", timeout=15000)

alert = page.locator(".MuiAlert-root").first
text  = await alert.inner_text()
await shot(page, "02-rejestracja-odrzucona")
await alert.screenshot(path=str(SHOTS / "03-komunikat.png"))
display(Image(str(SHOTS / "03-komunikat.png"), width=520))

await browser.close(); await pw.stop()

print(text)
check("komunikat ma sekcję e-mail i sekcję password", "e-mail" in text and "password" in text, text)
check("sekcja e-mail niesie zdanie z serwera", "domain" in text.lower(), text)
check("kod hasła niesie parametr polityki", "min length not met: 5" in text, text)

## 6. Sprzątanie i podsumowanie

In [ ]:
print(psql(f"DELETE FROM security_settings WHERE name = '{KEY}';"))
OVERRIDE.unlink(missing_ok=True)
r = report()
check("po sprzątaniu w mocy jest domyślne 5, bez odrzuconych", r["value"] == 5 and not r.get("rejected"), r)

ok = sum(1 for _, passed, _ in CHECKS if passed)
rows = "".join(f"<tr><td style='color:{'#137333' if p else '#c5221f'};padding:2px 10px 2px 0'><b>{'PASS' if p else 'FAIL'}</b></td>"
               f"<td style='padding:2px 10px 2px 0'>{n}</td><td style='color:#666'>{d if not p else ''}</td></tr>"
               for n, p, d in CHECKS)
display(HTML(f"<h3>{ok}/{len(CHECKS)} sprawdzeń przeszło</h3>"
             f"<table style='font-family:monospace;font-size:13px'>{rows}</table>"
             f"<p>Zrzuty: {', '.join(sorted(p.name for p in SHOTS.glob('*.png')))}</p>"))